# HDFC Bank: Fraud Risk Analysis - Part 4

**Objective:** MLflow Integration & Advanced Model Shootout

In this notebook, we wrap our training process in MLflow. We now use our `RigorousFeatureSelector` to scientifically pick the best features, rather than relying on human guesswork!

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

from fraudguard.data.ingestion import load_bank_data, split_temporal
from fraudguard.features.engineering import build_feature_pipeline
from fraudguard.features.selection import RigorousFeatureSelector
from fraudguard.models.training import train_logistic_regression, train_lightgbm, train_xgboost
from fraudguard.models.evaluation import evaluate_model

# Connect to the local MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("FraudGuard_V1")

## 1. Data Preparation & Dynamic Feature Selection

In [ ]:
# 1. Load Data
data_dir = Path.cwd().parent / "data" / "raw"
df = load_bank_data(data_dir)

# 2. Split Temporally
df_train, df_test = split_temporal(df, test_ratio=0.2)
X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud'].values
X_test = df_test.drop(columns=['isFraud'])
y_test = df_test['isFraud'].values

# 3. DYNAMIC Feature Selection (Trusting the Math!)
print("Running Rigorous Feature Selection on Training Data...")
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
selector = RigorousFeatureSelector(max_missing_ratio=0.90, top_n_features=15)
selector.fit(X_train[numeric_cols], y_train)

best_numeric_features = selector.selected_features_
print(f"\nScientifically Selected Features:\n{best_numeric_features}")

categorical_features = ['ProductCD', 'card4', 'card6']

# 4. Build and Apply Pipeline using ONLY the mathematically proven features
pipeline = build_feature_pipeline(best_numeric_features, categorical_features)
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

## 2. Train & Log Logistic Regression (Baseline)

In [ ]:
with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    
    lr_model = train_logistic_regression(X_train_processed, y_train, class_weight='balanced')
    lr_probs = lr_model.predict_proba(X_test_processed)[:, 1]
    lr_metrics = evaluate_model(y_test, lr_probs, threshold=0.5)
    mlflow.log_metrics(lr_metrics)
    
    from sklearn.pipeline import Pipeline
    full_deployable_model = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('classifier', lr_model)
    ])
    mlflow.sklearn.log_model(full_deployable_model, "model")
    print("Logistic Regression run logged to MLflow.")

## 3. Train & Log LightGBM

In [ ]:
with mlflow.start_run(run_name="LightGBM_Optimized"):
    mlflow.log_param("model_type", "LightGBM")
    
    lgb_model = train_lightgbm(X_train_processed, y_train, X_val=X_test_processed, y_val=y_test)
    lgb_probs = lgb_model.predict_proba(X_test_processed)[:, 1]
    lgb_metrics = evaluate_model(y_test, lgb_probs, threshold=0.5)
    mlflow.log_metrics(lgb_metrics)
    
    full_deployable_model = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('classifier', lgb_model)
    ])
    mlflow.sklearn.log_model(full_deployable_model, "model")
    print("LightGBM run logged to MLflow.")

## 4. Train & Log XGBoost

In [ ]:
with mlflow.start_run(run_name="XGBoost_Industry_Standard"):
    mlflow.log_param("model_type", "XGBoost")
    
    xgb_model = train_xgboost(X_train_processed, y_train, X_val=X_test_processed, y_val=y_test)
    xgb_probs = xgb_model.predict_proba(X_test_processed)[:, 1]
    xgb_metrics = evaluate_model(y_test, xgb_probs, threshold=0.5)
    mlflow.log_metrics(xgb_metrics)
    
    full_deployable_model = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('classifier', xgb_model)
    ])
    mlflow.sklearn.log_model(full_deployable_model, "model")
    print("XGBoost run logged to MLflow.")
    
print("\nAll runs complete! Go check your MLflow UI at http://127.0.0.1:5000 !")